In [15]:
import pandas as pd

KEYS = ["question_id", "compute_level", "sample_index"]

PATHS = {
         "df1": "combined_labeled_unique_clean.csv",
         "df2": "combined_labeled_unique.csv"
         }
PATH5 = "r1-llama8b_raw_results_segmented.csv"

SAVE_CLEAN_PATHS = True   # also save a cleaned copy of each path as <name>_clean.csv
FIX_FLOAT_KEYS = True   # "1.0" -> "1" in the key columns (happens when a file was saved with NaNs in a numeric column)

OUT_COMBINED = "combined_labeled_unique.csv"
OUT_PATHS_NOT_IN_DF5 = "in_paths_not_in_df5.csv"
OUT_DF5_NOT_IN_COMBINED = "df5_not_in_combined.csv"

def save_csv_checked(df, path):
    with open(path, "w", encoding="utf-8", newline="") as f:
        df.to_csv(f, index=False)
    with open(path, "rb") as f:
        f.read().decode("utf-8")   # fails right away if the file is not valid UTF-8
    print(f"saved {path} | {len(df):,} rows | valid UTF-8")

def load(path):
    df = pd.read_csv(path, encoding="utf-8-sig", dtype={k: str for k in KEYS})
    miss = [k for k in KEYS if k not in df.columns]
    if miss:
        raise ValueError(f"{path}: missing columns {miss}")
    df[KEYS] = df[KEYS].apply(lambda s: s.str.strip())
    if FIX_FLOAT_KEYS:
        df[KEYS] = df[KEYS].apply(lambda s: s.str.replace(r"^(-?\d+)\.0+$", r"\1", regex=True))
    return df.reset_index(drop=True)

raw = {n: load(p) for n, p in PATHS.items()}
df5 = load(PATH5)

for n, d in {**raw, "df5": df5}.items():
    print(f"{n}: {len(d):,} rows | unique keys: {d[KEYS].drop_duplicates().shape[0]:,} | NaN keys: {int(d[KEYS].isna().any(axis=1).sum())}")
    if n != "df5":
        if "rtl_status" in d.columns:
            print("   rtl_status:", d["rtl_status"].value_counts(dropna=False).to_dict())
        else:
            print("   WARNING: no rtl_status column, none of its rows will be kept")

# ---- cleaning: drop broken rows (fragments of rows that were split while saving) ----
# df5 is clean, so a row whose key is not in df5 is a fragment. Rows with a valid key but a
# broken / empty rtl_status are truncated rows: they are dropped too and will show up as "not done".
k5 = pd.MultiIndex.from_frame(df5[KEYS])
VALID_PREFIXES = ("pending", "skipped", "failed")

SCHEMA = [c for c in raw["df1"].columns if not str(c).startswith("Unnamed")]   # df1 is the clean reference
for n, d in raw.items():
    missing = [c for c in SCHEMA if c not in d.columns]
    if missing:
        raise ValueError(f"{n} is missing columns that df1 has: {missing[:5]}")

report = []
for n in list(raw):
    d = raw[n]
    in5 = pd.MultiIndex.from_frame(d[KEYS]).isin(k5)
    if "rtl_status" in d.columns:
        st = d["rtl_status"]
        ok = in5 & (st.eq("labeled") | st.fillna("").str.startswith(VALID_PREFIXES))
    else:
        ok = in5
    report.append({"file": n, "rows": len(d),
                   "fragments (key not in df5)": int((~in5).sum()),
                   "valid key, broken status": int((in5 & ~ok).sum()),
                   "kept": int(ok.sum())})
    extra = [c for c in d.columns if c not in SCHEMA]                  # "Unnamed: N" columns created by the broken rows
    extra_used = [c for c in extra if d.loc[ok, c].notna().any()]      # extra columns that still hold data in the kept rows
    if extra_used:
        print(f"WARNING {n}: {len(extra_used)} extra columns have data in kept rows, e.g. {extra_used[:5]}")
    report[-1]["extra columns dropped"] = len(extra)
    raw[n] = d.loc[ok, SCHEMA].reset_index(drop=True)
    if SAVE_CLEAN_PATHS:
        save_csv_checked(raw[n], PATHS[n].replace(".csv", "_clean.csv"))
display(pd.DataFrame(report))

df1: 9,030 rows | unique keys: 9,030 | NaN keys: 0
   rtl_status: {'labeled': 7680, 'skipped: no thinking text': 1350}
df2: 9,388 rows | unique keys: 9,388 | NaN keys: 0
   rtl_status: {'labeled': 8038, 'skipped: no thinking text': 1350}
df5: 9,450 rows | unique keys: 9,450 | NaN keys: 0
saved combined_labeled_unique_clean_clean.csv | 9,030 rows | valid UTF-8
saved combined_labeled_unique_clean.csv | 9,388 rows | valid UTF-8


,file,rows,fragments (key not in df5),"valid key, broken status",kept,extra columns dropped
0,df1,9030,0,0,9030,0
1,df2,9388,0,0,9388,0


In [16]:
all_rows = pd.concat([d.assign(_src=n) for n, d in raw.items()], ignore_index=True)
if "rtl_status" not in all_rows.columns:
    raise ValueError("rtl_status is missing from all paths")

is_labeled = all_rows["rtl_status"].eq("labeled")
labeled_rows = all_rows[is_labeled]

# one copy per key, labeled only (copies are taken in path order df1 -> df4)
combined = (labeled_rows.drop_duplicates(KEYS, keep="first")
                        .drop(columns="_src")
                        .reset_index(drop=True))

assert combined["rtl_status"].eq("labeled").all()
assert not combined.duplicated(KEYS).any()

print("rows in the 4 paths           :", f"{len(all_rows):,}")
print("dropped (rtl_status != labeled):", f"{(~is_labeled).sum():,}")
print("dropped (duplicate labeled)    :", f"{len(labeled_rows) - len(combined):,}")
print("combined (unique, labeled)     :", f"{len(combined):,}")

save_csv_checked(combined, OUT_COMBINED)

rows in the 4 paths           : 18,418
dropped (rtl_status != labeled): 2,700
dropped (duplicate labeled)    : 7,680
combined (unique, labeled)     : 8,038
saved combined_labeled_unique.csv | 8,038 rows | valid UTF-8


In [17]:
k5 = pd.MultiIndex.from_frame(df5[KEYS])
k_comb = pd.MultiIndex.from_frame(combined[KEYS])
k_all = pd.MultiIndex.from_frame(all_rows[KEYS])

print("df5: rows", f"{len(df5):,}", "| unique keys", f"{df5[KEYS].drop_duplicates().shape[0]:,}")
print("combined: rows", f"{len(combined):,}")
print("combined keys not in df5:", f"{(~k_comb.isin(k5)).sum():,}")
print("(any status) path rows not in df5:", f"{(~k_all.isin(k5)).sum():,}")

paths_not_in_df5 = (all_rows.loc[~k_all.isin(k5), KEYS + ["rtl_status", "_src"]]
                            .drop_duplicates(KEYS + ["_src"])
                            .sort_values(KEYS, kind="stable"))
print("\nin paths but not in df5, by file / status:")
print(paths_not_in_df5.groupby(["_src", "rtl_status"], dropna=False).size().to_string() if len(paths_not_in_df5) else "   none")
save_csv_checked(paths_not_in_df5, OUT_PATHS_NOT_IN_DF5)
display(paths_not_in_df5)

df5: rows 9,450 | unique keys 9,450
combined: rows 8,038
combined keys not in df5: 0
(any status) path rows not in df5: 0

in paths but not in df5, by file / status:
   none
saved in_paths_not_in_df5.csv | 0 rows | valid UTF-8


,question_id,compute_level,sample_index,rtl_status,_src


In [18]:
def norm_index(df):
    k = df[KEYS].astype(str).apply(
        lambda s: s.str.strip().str.lower().str.replace(r"^(-?\d+)\.0+$", r"\1", regex=True))
    return pd.MultiIndex.from_frame(k)

raw_missing = int((~k_all.isin(k5)).sum())
norm_missing = int((~norm_index(all_rows).isin(norm_index(df5))).sum())
print("path rows not in df5 (exact match)     :", raw_missing)
print("path rows not in df5 (after normalizing):", norm_missing)
if norm_missing < raw_missing:
    print("-> part of the mismatch is formatting only (case / spaces / '1.0' vs '1')")

print("\nexamples (path key vs df5 keys with the same question_id):")
for _, r in paths_not_in_df5.drop_duplicates(KEYS).head(5).iterrows():
    print("path:", tuple(r[KEYS]), "| file:", r["_src"], "| status:", r["rtl_status"])
    same_q = df5.loc[df5["question_id"] == r["question_id"], KEYS].drop_duplicates().head(5)
    print(same_q.to_string(index=False) if len(same_q) else "   question_id not found in df5")

path rows not in df5 (exact match)     : 0
path rows not in df5 (after normalizing): 0

examples (path key vs df5 keys with the same question_id):


In [19]:
not_in_combined = ~k5.isin(k_comb)
df5_not_in_combined = df5[not_in_combined].copy()

k_unlabeled = pd.MultiIndex.from_frame(all_rows.loc[~is_labeled, KEYS].drop_duplicates())
print("df5 rows not in combined:", f"{len(df5_not_in_combined):,}")
print("   present in paths but never labeled:", f"{(not_in_combined & k5.isin(k_unlabeled)).sum():,}")
print("   not in any path                   :", f"{(not_in_combined & ~k5.isin(k_unlabeled)).sum():,}")

save_csv_checked(df5_not_in_combined, OUT_DF5_NOT_IN_COMBINED)
display(df5_not_in_combined)

df5 rows not in combined: 1,412
   present in paths but never labeled: 1,350
   not in any path                   : 62
saved df5_not_in_combined.csv | 1,412 rows | valid UTF-8


,question_id,benchmark,difficulty_tier,task_type,compute_level,sample_index,raw_model_output,thinking_text,gold,is_correct,tokens_used,thinking_tokens,output_tokens,hit_response_marker,latency_s,was_truncated,hit_marker_empty_output
0,GSM8K_0,GSM8K,easy,math,0,3,**Solution:**\n\nLet's break down the problem ...,NaN,18,True,223,0,222,True,6.598637,False,False
1,GSM8K_0,GSM8K,easy,math,0,0,**Solution:**\n\nLet's break down the problem ...,NaN,18,True,223,0,222,True,7.257009,False,False
2,GSM8K_0,GSM8K,easy,math,0,1,**Solution:**\n\nLet's break down the problem ...,NaN,18,True,353,0,352,True,10.817345,False,False
3,GSM8K_0,GSM8K,easy,math,0,4,Let's break down the problem step by step to d...,NaN,18,True,330,0,329,True,11.128524,False,False
5,GSM8K_0,GSM8K,easy,math,0,2,**Solution:**\n\nLet's break down the problem ...,NaN,18,True,376,0,375,True,12.901834,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9393,LiveCodeBench-v6_29,LiveCodeBench-v6,hard,coding,0,4,"To solve this problem, we need to transform a ...",NaN,"[{'input': '""egcfe""', 'output': '""efcfe""', 'te...",True,588,0,587,True,106.830651,False,False
9394,LiveCodeBench-v6_29,LiveCodeBench-v6,hard,coding,0,1,"To solve this problem, we need to transform a ...",NaN,"[{'input': '""egcfe""', 'output': '""efcfe""', 'te...",False,859,0,858,True,208.850793,False,False
9400,LiveCodeBench-v6_29,LiveCodeBench-v6,hard,coding,0,3,"To solve this problem, we need to transform a ...",NaN,"[{'input': '""egcfe""', 'output': '""efcfe""', 'te...",False,763,0,762,True,231.341992,False,False
9425,LiveCodeBench-v6_27,LiveCodeBench-v6,hard,coding,8192,2,"To solve this problem, we need to find the min...","⟦S1⟧Okay, so I've got this problem to solve. ⟦...","[{'input': '5\n4', 'output': '18', 'testtype':...",False,8588,8192,395,True,1352.476039,False,False


In [20]:
# ===== check: combined + the rest == df5 =====
k5 = pd.MultiIndex.from_frame(df5[KEYS])
k_comb = pd.MultiIndex.from_frame(combined[KEYS])
k_rest = pd.MultiIndex.from_frame(df5_not_in_combined[KEYS])
k_unlabeled = pd.MultiIndex.from_frame(all_rows.loc[~is_labeled, KEYS].drop_duplicates())

never_labeled = ~k5.isin(k_comb) & k5.isin(k_unlabeled)    # in the paths but not labeled
not_in_paths  = ~k5.isin(k_comb) & ~k5.isin(k_unlabeled)   # in no path at all

checks = {
    "df5 keys are unique": not k5.duplicated().any(),
    "combined keys are unique": not k_comb.duplicated().any(),
    "every combined key is in df5": bool(k_comb.isin(k5).all()),
    "combined and df5_not_in_combined do not overlap": not k_rest.isin(k_comb).any(),
    "combined + df5_not_in_combined == df5 (row count)": len(combined) + len(df5_not_in_combined) == len(df5),
    "combined keys + rest keys == df5 keys (exact sets)": set(k_comb) | set(k_rest) == set(k5),
}
for name, ok in checks.items():
    print("OK  " if ok else "FAIL", name)

print(f"\ncombined        : {len(combined):>7,}")
print(f"never labeled   : {int(never_labeled.sum()):>7,}   (in the paths, not labeled)")
print(f"not in any path : {int(not_in_paths.sum()):>7,}")
print(f"sum             : {len(combined) + int(never_labeled.sum()) + int(not_in_paths.sum()):>7,}   df5: {len(df5):,}")

assert all(checks.values()), "mismatch, see the FAIL lines above"

OK   df5 keys are unique
OK   combined keys are unique
OK   every combined key is in df5
OK   combined and df5_not_in_combined do not overlap
OK   combined + df5_not_in_combined == df5 (row count)
OK   combined keys + rest keys == df5 keys (exact sets)

combined        :   8,038
never labeled   :   1,350   (in the paths, not labeled)
not in any path :      62
sum             :   9,450   df5: 9,450


In [ ]:
# ===== final verification of the two outputs: combined + df5_not_in_combined =====
def keyset(df):
    return set(pd.MultiIndex.from_frame(df[KEYS]))

def read_back(path):
    with open(path, "rb") as f:
        f.read().decode("utf-8")                      # raises if not valid UTF-8
    return pd.read_csv(path, dtype={k: str for k in KEYS})

disk_comb = read_back(OUT_COMBINED)
disk_rest = read_back(OUT_DF5_NOT_IN_COMBINED)

k5, kc, kr = keyset(df5), keyset(combined), keyset(df5_not_in_combined)
rest_expected = df5[~pd.MultiIndex.from_frame(df5[KEYS]).isin(pd.MultiIndex.from_frame(combined[KEYS]))]

checks = {
    # --- combined (in memory) ---
    "combined: columns == df1 schema (no Unnamed)": list(combined.columns) == SCHEMA,
    "combined: every row is labeled": bool(combined["rtl_status"].eq("labeled").all()),
    "combined: no NaN keys": not combined[KEYS].isna().any().any(),
    "combined: keys are unique": not combined.duplicated(KEYS).any(),
    "combined: every key is in df5": kc <= k5,
    # --- df5_not_in_combined (in memory) ---
    "rest: same columns as df5": list(df5_not_in_combined.columns) == list(df5.columns),
    "rest: keys are unique": not df5_not_in_combined.duplicated(KEYS).any(),
    "rest: every key is in df5": kr <= k5,
    "rest: identical to df5 rows that are not in combined": df5_not_in_combined.reset_index(drop=True).equals(rest_expected.reset_index(drop=True)),
    # --- the two together ---
    "no key in both": not (kc & kr),
    "combined + rest == df5 (row count)": len(combined) + len(df5_not_in_combined) == len(df5),
    "combined + rest == df5 (exact key sets)": (kc | kr) == k5,
    # --- files on disk ---
    "disk combined: shape == memory": disk_comb.shape == combined.shape,
    "disk combined: same keys": keyset(disk_comb) == kc,
    "disk combined: no Unnamed columns": not any(str(c).startswith("Unnamed") for c in disk_comb.columns),
    "disk rest: shape == memory": disk_rest.shape == df5_not_in_combined.shape,
    "disk rest: same keys": keyset(disk_rest) == kr,
    "disk rest: no Unnamed columns": not any(str(c).startswith("Unnamed") for c in disk_rest.columns),
}
for name, ok in checks.items():
    print("OK  " if ok else "FAIL", name)

print(f"\ncombined {len(combined):,} x {combined.shape[1]} cols | rest {len(df5_not_in_combined):,} x {df5_not_in_combined.shape[1]} cols | df5 {len(df5):,}")
assert all(checks.values()), "some checks failed, see the FAIL lines above"
print("\nALL CHECKS PASSED")

OK   combined: columns == df1 schema (no Unnamed)
OK   combined: every row is labeled
OK   combined: no NaN keys
OK   combined: keys are unique
OK   combined: every key is in df5
OK   rest: same columns as df5
OK   rest: keys are unique
OK   rest: every key is in df5
OK   rest: identical to df5 rows that are not in combined
OK   no key in both
OK   combined + rest == df5 (row count)
OK   combined + rest == df5 (exact key sets)
OK   disk combined: shape == memory
OK   disk combined: same keys
OK   disk combined: no Unnamed columns
OK   disk rest: shape == memory
OK   disk rest: same keys
OK   disk rest: no Unnamed columns

combined 8,038 x 20 cols | rest 1,412 x 17 cols | df5 9,450

ALL CHECKS PASSED


In [ ]:
SKIP_STATUS = "skipped: no thinking text"

print("compute_level values in df5_not_in_combined:")
print(df5_not_in_combined["compute_level"].value_counts(dropna=False).to_string())

is_zero = pd.to_numeric(df5_not_in_combined["compute_level"], errors="coerce").eq(0)
zero_rows = df5_not_in_combined[is_zero]
print(f"\nrows with compute_level == 0: {len(zero_rows):,} of {len(df5_not_in_combined):,}")

# sanity: these rows are supposed to have no thinking text
if "thinking_text" in zero_rows.columns:
    has_text = zero_rows["thinking_text"].fillna("").astype(str).str.strip().ne("")
    print("   of them WITH thinking_text:", int(has_text.sum()))

lost = [c for c in zero_rows.columns if c not in combined.columns]
if lost:
    print("   df5 columns that combined does not have (not copied):", lost)

moved = zero_rows.reindex(columns=combined.columns)     # label columns stay empty
moved["rtl_status"] = SKIP_STATUS

combined = pd.concat([combined, moved], ignore_index=True)
df5_not_in_combined = df5_not_in_combined[~is_zero].reset_index(drop=True)

print(f"\ncombined: {len(combined):,} rows ({int(combined['rtl_status'].eq(SKIP_STATUS).sum()):,} with '{SKIP_STATUS}')")
print(f"df5_not_in_combined: {len(df5_not_in_combined):,} rows")

save_csv_checked(combined, OUT_COMBINED)
save_csv_checked(df5_not_in_combined, OUT_DF5_NOT_IN_COMBINED)

compute_level values in df5_not_in_combined:
compute_level
0       1350
8192      62

rows with compute_level == 0: 1,350 of 1,412
   of them WITH thinking_text: 0

combined: 9,388 rows (1,350 with 'skipped: no thinking text')
df5_not_in_combined: 62 rows
saved combined_labeled_unique.csv | 9,388 rows | valid UTF-8
saved df5_not_in_combined.csv | 62 rows | valid UTF-8


In [ ]:
import pandas as pd
nott = pd.read_csv("df5_not_in_combined.CSV")

In [ ]:
nott.shape

(62, 17)